# Nearest-neighbor anomaly detection on MVTec `pill`

Pool all training features from the `good` class into a memory bank, score test images by their L2 distance to the k nearest bank vectors. This notebook reproduces the `pill` result shown in the README.

**Method summary.** No distributional assumption on normal data. Every training feature vector (from every image, every spatial location) goes into a bank of shape `(M, C)`. At inference, a test feature vector at position `(h, w)` is compared to *all* bank entries, and its anomaly score is the mean distance to its top-k nearest neighbors.

Reference: Roth et al., *Towards Total Recall in Industrial Anomaly Detection* (PatchCore), CVPR 2022. Simplified: no coreset subsampling, just random sampling of the bank.

In [1]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from src.dataset import MVTecDataset, IMAGENET_MEAN, IMAGENET_STD
from src.features import FeatureExtractor
from src.methods import NearestNeighborAD
from src.visualize import upsample_and_smooth, plot_anomaly

CATEGORY = "pill"
DATA_ROOT = "../data/mvtec"
IMAGE_SIZE = 224
BATCH_SIZE = 16
K = 3
MAX_BANK_SIZE = 20_000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## Data

The `pill` category has ~267 training images (all `good`) and a test set with `good` plus seven defect types (`color`, `combined`, `contamination`, `crack`, `faulty_imprint`, `pill_type`, `scratch`).

In [2]:
tfm = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = MVTecDataset(DATA_ROOT, CATEGORY, is_train=True, transform=tfm)
test_ds  = MVTecDataset(DATA_ROOT, CATEGORY, is_train=False, transform=tfm)

print(f"train : {len(train_ds)} images (all good)")
print(f"test  : {len(test_ds)} images across defect types: {test_ds.defect_types()}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)

train : 267 images (all good)
test  : 167 images across defect types: ['color', 'combined', 'contamination', 'crack', 'faulty_imprint', 'good', 'pill_type', 'scratch']


## Feature extractor

Same as PaDiM: frozen ResNet-18 truncated at `layer3`. Feature map shape `(256, 14, 14)` per image.

In [3]:
extractor = FeatureExtractor(stop_layer="layer3").to(device)
print(f"output channels: {extractor.num_channels}")
print(f"trainable params: {sum(p.numel() for p in extractor.parameters() if p.requires_grad)}")

output channels: 256
trainable params: 0


## Build the memory bank

Extract features for every training image, flatten spatial dimensions, pool into one bank. Natural size before subsampling: `267 imgs x 14 x 14 = 52,332` vectors of 256 dimensions each. We cap at 20,000 randomly-sampled vectors, a coarser approximation of PatchCore's coreset algorithm.

In [ ]:
%%time
model = NearestNeighborAD(k=K, max_bank_size=MAX_BANK_SIZE)
model.fit(extractor, train_loader, device)
print("bank shape:", model.bank.shape)
print(f"memory footprint: {model.bank.numel() * 4 / 1e6:.1f} MB (float32)")

## Score and visualize test images

Pick one image per defect type, compute the kNN score map (per-pixel mean distance to the 3 nearest bank vectors), upsample, smooth, overlay.

In [ ]:
seen = {}
for i, path in enumerate(test_ds.image_paths):
    defect = path.split("/")[-2]
    if defect != "good" and defect not in seen:
        seen[defect] = i
    if len(seen) == 3:
        break

print("showing indices:", seen)

for defect, idx in seen.items():
    img, lbl = test_ds[idx]
    fmap = extractor(img.unsqueeze(0).to(device)).squeeze(0)
    score = model.score(fmap)
    heatmap = upsample_and_smooth(score, out_size=IMAGE_SIZE, sigma=4.0)

    fig = plot_anomaly(img, heatmap, label=lbl, method_name=f"kNN on pill/{defect}")
    fig.show()

## Analysis

kNN works well on `pill` because the category has natural object-level variation between training samples (pose, lighting, imprint alignment). A per-location Gaussian model diffuses across that variation, but a memory bank retains every training feature as a discrete point. A colored speck on a test pill has no close neighbor in the bank because no training pill has anything like it locally, so the distance is high in the right place.

The failure mode of kNN shows up on repetitive textures where a defective patch can still find *some* neighbor in the bank that looks vaguely similar (e.g. bent grid wires being loosely similar to normal wires with a slight bump). See `02_padim_baseline.ipynb` for the `grid` case where PaDiM localizes broken wires cleanly and kNN barely reacts.

**On the memory bank cap.** With 267 pill training images, the natural bank size is 52k vectors. Capping at 20k random samples drops fitting/scoring by ~2.5x with no visible change in heatmap quality for this category, because the training features are highly redundant. The full PatchCore paper uses a greedy k-center coreset instead of random sampling, keeping the ~10% most representative vectors and reaching ~99% AUROC on MVTec AD.